# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rimlazrek1/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup (Local)

In [7]:
import os
from pathlib import Path

import duckdb

ROOT = Path.cwd()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    import getpass
    print("Tip: pip install python-dotenv")

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HF READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("Connected.")

Connected.


## 1. My rule and its reason codes

**Lane 4 — CTR / Engagement Opportunity Scoring**

**Data contract (ML-04):** warehouse **March 2026** (`month=2026-03`), one row per page, `imp_mar >= 100` for feature build, real position only. Label and **ranked queue** use the **`imp_mar >= 500` floor** from the contract — see `w03_data_contract.ipynb`.

### My rule

> Rank pages **higher** when **`imp_mar >= 500`** and their **`ctr_mar` is below the median for their `position_tier`**. Score = `ctr_gap` (tier median − page CTR). **Tie-break:** higher `imp_mar` first (more stake). A reviewer opens the top of that list first.

### Two signal checks

Before scoring, we check two ideas the rule depends on. Each mirrors a FlyRank-style flag from the session docs (e.g. `low_ctr_visible_page` = impressions ≥ 500, position 1–20, CTR < 0.5).


| Signal | What we check | FlyRank idea it mirrors |
|---|---|---|
| **1. CTR vs position** | Does CTR change by `position_tier` | Similar to their `low_ctr_visible_page` flag — “visible page, weak CTR.” |
| **2. Volume** | Are low-impression pages noisier? | Similar to their impression floor (with few impressions, CTR jumps around). |

### Reason codes 
*Labels that explain why a page scored (first match wins)*

| Reason code | When | In exported CSV? |
|---|---|---|
| `high_visibility_ctr_gap` | below tier median **and** `imp_mar >= 500` | Yes — underperformers in the queue |
| `ctr_below_tier_median` | below tier median but `imp_mar < 500` | No — below queue floor |
| `general_ctr_monitor` | at or above tier median (`imp_mar >= 500`) | Yes — ranked low, not priority picks |

### Signal verdicts
*Giving each a one-word verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE*

**Signal 1 — CTR vs position:** `CONFIRMED`  
Weighted CTR falls from 0.41% on `top_3` to 0.04% on `deep` (n = 8,295 and 3,949 pages), so CTR must be compared within `position_tier`, not site-wide.

**Signal 2 — Volume:** `CONFIRMED`  
Low-impression bands are noisier: 74% zero-click pages in `100-299` (n = 26,775) vs 3% in `3000+` (n = 22,157), so a minimum impression floor is needed before we trust CTR.    
   
*n = how many pages are in that bucket.*


In [8]:
import pandas as pd

TIER_ORDER = ["top_3", "page_1", "striking", "page_3_5", "deep"]
IMP_BANDS = [100, 300, 500, 1000, 3000, float("inf")]
IMP_LABELS = ["100-299", "300-499", "500-999", "1000-2999", "3000+"]

features = con.sql(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_mar,
            SUM(gsc_clicks) AS clk_mar,
            AVG(NULLIF(gsc_avg_position, 0)) AS pos_avg_mar
        FROM {FACT_MAR}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
    ),
    scored AS (
        SELECT
            *,
            CASE WHEN imp_mar > 0 THEN 100.0 * clk_mar / imp_mar END AS ctr_mar,
            CASE
                WHEN pos_avg_mar <= 3 THEN 'top_3'
                WHEN pos_avg_mar <= 10 THEN 'page_1'
                WHEN pos_avg_mar <= 20 THEN 'striking'
                WHEN pos_avg_mar <= 50 THEN 'page_3_5'
                ELSE 'deep'
            END AS position_tier
        FROM daily
        WHERE pos_avg_mar > 0
    ),
    labeled AS (
        SELECT
            s.*,
            MEDIAN(ctr_mar) OVER (PARTITION BY position_tier) AS tier_median_ctr,
            CASE
                WHEN ctr_mar < MEDIAN(ctr_mar) OVER (PARTITION BY position_tier)
                     AND imp_mar >= 500
                THEN 1
                ELSE 0
            END AS is_ctr_underperformer
        FROM scored s
    )
    SELECT * FROM labeled
""").df()

print(f"Lane slice n = {len(features):,}")

print("\nSIGNAL 1 — CTR vs position (by position_tier)")
sig1 = (
    features.groupby("position_tier", observed=True)
    .agg(
        n=("content_hash_id", "count"),
        sum_imp=("imp_mar", "sum"),
        sum_clk=("clk_mar", "sum"),
    )
    .assign(weighted_ctr_mar=lambda d: 100.0 * d["sum_clk"] / d["sum_imp"])
    .drop(columns=["sum_imp", "sum_clk"])
    .reindex(TIER_ORDER)
    .reset_index()
)
print(sig1.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nSIGNAL 2 — Volume (by impression band)")
features["imp_band"] = pd.cut(
    features["imp_mar"], bins=IMP_BANDS, labels=IMP_LABELS, right=False
)
sig2 = (
    features.groupby("imp_band", observed=True)
    .agg(
        n=("content_hash_id", "count"),
        share_zero_clicks=("ctr_mar", lambda s: (s == 0).mean()),
    )
    .reset_index()
)
print(sig2.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

Lane slice n = 101,441

SIGNAL 1 — CTR vs position (by position_tier)
position_tier     n  weighted_ctr_mar
        top_3  8295            0.4067
       page_1 46531            0.3232
     striking 21738            0.3052
     page_3_5 20928            0.1361
         deep  3949            0.0356

SIGNAL 2 — Volume (by impression band)
 imp_band     n  share_zero_clicks
  100-299 26775             0.7393
  300-499 12742             0.5642
  500-999 16866             0.3801
1000-2999 22901             0.1624
    3000+ 22157             0.0301


## 2. Build the ranked queue (writes the CSV)

**Score:** `baseline_score = ctr_gap` (tier median CTR − page CTR).  
**Population:** only `imp_mar >= 500` pages get a rank (same floor as the label).  
**Sort:** `ctr_gap` desc, then `imp_mar` desc to break ties among zero-click rows.

*Writes `work/outputs/baseline_action_score.csv`.*

In [9]:
import numpy as np

OUT = ROOT / "work" / "outputs" / "baseline_action_score.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

IMP_FLOOR = 500
queue = features.copy()
queue["ctr_gap"] = queue["tier_median_ctr"] - queue["ctr_mar"]
queue["baseline_score"] = queue["ctr_gap"]
queue["below_tier_median"] = (queue["ctr_mar"] < queue["tier_median_ctr"]).astype(int)


def reason_code(row) -> str:
    if row["imp_mar"] >= IMP_FLOOR and row["below_tier_median"]:
        return "high_visibility_ctr_gap"
    if row["below_tier_median"]:
        return "ctr_below_tier_median"
    return "general_ctr_monitor"


def action_label(code: str) -> str:
    if code == "high_visibility_ctr_gap":
        return "refresh_and_review_ctr"
    if code == "ctr_below_tier_median":
        return "review_ctr"
    return "monitor"


# Rank eligible pool only (ML-04 contract floor)
eligible = queue[queue["imp_mar"] >= IMP_FLOOR].copy()
eligible = eligible.sort_values(["baseline_score", "imp_mar"], ascending=[False, False])
eligible["baseline_rank"] = np.arange(1, len(eligible) + 1)
eligible["reason_code"] = eligible.apply(reason_code, axis=1)
eligible["action"] = eligible["reason_code"].map(action_label)

export_cols = [
    "baseline_rank", "content_hash_id", "client_hash_id",
    "imp_mar", "ctr_mar", "pos_avg_mar", "position_tier",
    "baseline_score", "reason_code", "action",
]
eligible.to_csv(OUT, index=False, columns=export_cols)
print(f"Wrote {len(eligible):,} eligible rows (imp_mar >= {IMP_FLOOR}) → {OUT}")

queue = eligible  # downstream cells use the ranked eligible queue


Wrote 61,924 eligible rows (imp_mar >= 500) → c:\Users\rimla\Desktop\work_folder\flyrank-internship\work\outputs\baseline_action_score.csv


### Baseline metrics

**Population:** `imp_mar >= 500` (same as ranked queue)  
**Label** = `is_ctr_underperformer` (CTR below tier median and `imp_mar >= 500`)  
**Score** = `ctr_gap`  
**Metric** = Precision@K — of the top K pages our rule ranks, how many are underperformers?

On this eligible pool, Precision@K ≈ **1.0** is expected: `ctr_gap` uses the same ingredients as the label, so the top ranks are almost always true underperformers. Compare to **base rate** to see lift vs random picking.

In [10]:
import json
import numpy as np

LABEL = "is_ctr_underperformer"

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    k = min(k, len(order))
    return float(np.asarray(labels)[order[:k]].mean())

labels = queue[LABEL].to_numpy()
scores = queue["baseline_score"].to_numpy()
base_rate = float(labels.mean())

metrics = {
    "method": "baseline_rule",
    "label": LABEL,
    "slice": "month=2026-03",
    "population": f"imp_mar>={IMP_FLOOR}",
    "n": int(len(queue)),
    "base_rate": base_rate,
    "precision_at_10": precision_at_k(scores, labels, 10),
    "precision_at_20": precision_at_k(scores, labels, 20),
    "precision_at_50": precision_at_k(scores, labels, 50),
}

METRICS_OUT = ROOT / "work" / "outputs" / "baseline_metrics.json"
METRICS_OUT.write_text(json.dumps(metrics, indent=2))

print(f"Eligible n = {len(queue):,}")
print(f"Base rate (share {LABEL}=1): {base_rate:.3f}")
for k in (10, 20, 50):
    print(f"Precision@{k}: {metrics[f'precision_at_{k}']:.3f}")
print(f"Saved → {METRICS_OUT}")

Eligible n = 61,924
Base rate (share is_ctr_underperformer=1): 0.362
Precision@10: 1.000
Precision@20: 1.000
Precision@50: 1.000
Saved → c:\Users\rimla\Desktop\work_folder\flyrank-internship\work\outputs\baseline_metrics.json


## 3. Top-10 review

Hand review of the **top 10** (same method as a top-20 review — section 3 in the baseline skill).

For each row: **action**, **why it's here**, and **what would make it wrong**.

After the population gate (`imp_mar >= 500`) and `imp_mar` tie-break, every top-10 row is `high_visibility_ctr_gap` / `refresh_and_review_ctr` with **label = 1**. Many are `top_3` zero-click pages tied on max `ctr_gap` (~0.41%); higher impressions win the tie.

| Rank | action | reason_code | why it's here | what would make it wrong |
|---:|---|---|---|---|
| 1 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3`, **0% CTR**, **12,588** March impressions — max `ctr_gap` vs `top_3` median (~0.41%) | Featured snippet or zero-click SERP — title/meta edit won't move clicks |
| 2 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3`, **0% CTR**, **9,887** impressions — same max gap, high stake | Brand/nav query where clicks aren't expected |
| 3 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3`, **0% CTR**, **7,736** impressions — tied max `ctr_gap`, below tier median | Tracking gap — clicks exist but didn't roll into March GSC |
| 4 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3`, **0% CTR**, **6,827** impressions — high visibility + max gap | Intent mismatch; page ranks but users don't click |
| 5 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3`, **0% CTR**, **5,792** impressions — same zero-click `top_3` pattern | Seasonal March dip, not a page-specific CTR problem |
| 6 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3`, **0% CTR**, **4,742** impressions — still well above imp floor | Answer box satisfies query without a click |
| 7 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3`, **0% CTR**, **4,225** impressions — legitimate queue member after tie-break | Thin query mix — need query-level review (not in this slice) |
| 8 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3`, **0% CTR**, **4,202** impressions — same pattern as ranks 1–7 | Competitor SERP layout change, not on-page quality |
| 9 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3`, **0% CTR**, **4,047** impressions — high gap + imp ≥ 500 | Page is new — CTR hasn't stabilized yet |
| 10 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3`, **0% CTR**, **3,900** impressions — lowest imp in top 10 but still high visibility | Zero clicks at 3.9k imp may still be SERP-driven, not snippet-driven |

In [11]:
top10_cols = [
    "baseline_rank", "content_hash_id", "imp_mar", "ctr_mar",
    "position_tier", "reason_code", "action", "is_ctr_underperformer",
]
print(queue.head(10)[top10_cols].to_string(index=False))

 baseline_rank          content_hash_id  imp_mar  ctr_mar position_tier             reason_code                 action  is_ctr_underperformer
             1 content_fa17add7836d36c3  12588.0      0.0         top_3 high_visibility_ctr_gap refresh_and_review_ctr                      1
             2 content_d397987113cb84a0   9887.0      0.0         top_3 high_visibility_ctr_gap refresh_and_review_ctr                      1
             3 content_a27b382f00aa75c6   7736.0      0.0         top_3 high_visibility_ctr_gap refresh_and_review_ctr                      1
             4 content_83167156f76e33e5   6827.0      0.0         top_3 high_visibility_ctr_gap refresh_and_review_ctr                      1
             5 content_1bc8782404e3b132   5792.0      0.0         top_3 high_visibility_ctr_gap refresh_and_review_ctr                      1
             6 content_9cec93fc44a7ab41   4742.0      0.0         top_3 high_visibility_ctr_gap refresh_and_review_ctr                      1
      

## 4. Weak picks + leakage check

**Weak picks** *(1–2 ranks from the top 10 that look shaky and why)*:

1. **Rank 10** — ~3,900 impressions, 0% CTR on `top_3`. Legitimate by the rule, but lowest stake in the top 10; a reviewer might start at rank 1–3 (12k–10k imp) first.
2. **Rank 1** — ~12,588 impressions, 0% CTR. Highest priority by the rule, but an all-zero-click `top_3` cluster often means **SERP layout** (featured snippet, People Also Ask) rather than a fixable title/meta problem.

**Leakage check (plain words):**

- No product flags (`needs_ctr_fix`, `health_score`, …) — not in the data ✓
- No future-window columns — March 2026 only ✓
- No `trend_direction` / `trend_pct` in the score ✓
- `ctr_gap` is **part of the baseline rule**, not a model feature — OK for this hand rule; dropped for ML-08 models (see ML-04 trap) ✓
- Population gate `imp_mar >= 500` matches the label floor from ML-04 ✓

In [12]:
FORBIDDEN = {"trend_direction", "trend_pct", "health_score", "needs_ctr_fix", "priority_score"}
print("Forbidden cols present:", sorted(FORBIDDEN & set(queue.columns)) or "none (good)")

Forbidden cols present: none (good)


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.